<a href="https://colab.research.google.com/github/aiswaryapradeep5/filmrec1/blob/main/filmrec1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn

In [ ]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

# Load data
movies = pd.read_csv('ml-latest-small/movies.csv')
movies['genres_list'] = movies['genres'].str.split('|')

# Build feature matrix
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genres_list'])

# Compute cosine similarity
similarity = cosine_similarity(genre_matrix)
print("✅ Model ready!", similarity.shape)

In [ ]:
def recommend(title, n=6):
    idx = movies[movies['title'] == title].index[0]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    return movies.iloc[[i[0] for i in scores]][['title', 'genres']]

# Try it!
recommend("Toy Story (1995)")

In [ ]:
# Search for Matrix movies in the dataset
movies[movies['title'].str.contains('Matrix', case=False)]

In [ ]:
def recommend(title, n=6):
    match = movies[movies['title'] == title]

    if match.empty:
        # Show closest matches
        suggestions = movies[movies['title'].str.contains(title.split('(')[0].strip(), case=False)]['title'].tolist()
        print(f"❌ '{title}' not found. Did you mean one of these?")
        for s in suggestions[:5]:
            print(f"   👉 {s}")
        return

    idx = match.index[0]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    return movies.iloc[[i[0] for i in scores]][['title', 'genres']]

# Try again
recommend("Matrix")

In [ ]:
def search(keyword):
    results = movies[movies['title'].str.contains(keyword, case=False)]['title'].tolist()
    for r in results:
        print(r)

# Examples
search("Inception")
search("Avengers")
search("Batman")

In [ ]:
# Load ratings data
ratings = pd.read_csv('ml-latest-small/ratings.csv')

# Calculate average rating per movie
avg_ratings = ratings.groupby('movieId')['rating'].mean().reset_index()
avg_ratings.columns = ['movieId', 'avg_rating']

# Merge with movies
movies = movies.merge(avg_ratings, on='movieId', how='left')
movies['avg_rating'] = movies['avg_rating'].fillna(0)

# Rebuild feature matrix with ratings included
genre_matrix_df = pd.DataFrame(mlb.transform(movies['genres_list']), columns=mlb.classes_)
genre_matrix_df['avg_rating'] = movies['avg_rating'] / 5.0  # normalize to 0-1

# Recompute similarity
similarity = cosine_similarity(genre_matrix_df)
print("✅ Upgraded model ready with ratings!")

# Test it
recommend("Inception (2010)")

In [11]:
# Recommend based on what similar USERS watched
# not just genres
from sklearn.decomposition import TruncatedSVD

In [12]:
# Only recommend movies with 100+ ratings
popular = ratings.groupby('movieId').count()
popular_ids = popular[popular['rating'] >= 100].index
movies_popular = movies[movies['movieId'].isin(popular_ids)]

In [ ]:
!pip install ipywidgets
import ipywidgets as widgets
from IPython.display import display

dropdown = widgets.Dropdown(options=movies['title'].tolist(), description='Movie:')
button = widgets.Button(description='Recommend')

def on_click(b):
    display(recommend(dropdown.value))

button.on_click(on_click)
display(dropdown, button)

In [ ]:
# Check if movies dataframe is loaded correctly
print("Total movies:", len(movies))
print("\nSample titles:")
print(movies['title'].head(10).tolist())

In [ ]:
from ipywidgets import interact
import ipywidgets as widgets

@interact(movie=widgets.Dropdown(options=movies['title'].tolist()))
def show_recommendations(movie):
    result = recommend(movie)
    if result is not None:
        display(result)

In [ ]:
import requests
from IPython.display import Image, display, HTML
from ipywidgets import interact
import ipywidgets as widgets

TMDB_API_KEY = "a3402bc9bd4b901df12e1d0be2b76db8"  # 👈 replace with your key

def get_poster(title):
    clean_title = title.split('(')[0].strip()
    url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query={clean_title}"
    response = requests.get(url).json()
    if response['results']:
        poster_path = response['results'][0].get('poster_path')
        if poster_path:
            return f"https://image.tmdb.org/t/p/w200{poster_path}"
    return None

def recommend_with_posters(movie):
    result = recommend(movie)
    if result is None:
        return

    print(f"\n🎯 Because you liked: {movie}\n")

    # Build HTML with posters side by side
    html = "<div style='display:flex; flex-wrap:wrap; gap:15px;'>"
    for _, row in result.iterrows():
        poster_url = get_poster(row['title'])
        if poster_url:
            html += f"""
            <div style='text-align:center; width:150px;'>
                <img src='{poster_url}' width='140' style='border-radius:8px;'/>
                <p style='font-size:11px; margin-top:5px;'>{row['title']}</p>
            </div>"""
    html += "</div>"
    display(HTML(html))

# Interactive dropdown with posters
interact(recommend_with_posters,
         movie=widgets.Dropdown(options=movies['title'].tolist()))